# RSNA Knee 2.5D CNN — Smoke and Baseline

## What this notebook does

This notebook builds a compact, reproducible 2.5D convolutional neural network for all 12 RSNA knee abnormality targets.

1. **Stage 1** validates the Kaggle environment, competition files, and exact submission schema without recursively listing the 570 GB input tree.
2. **Stage 2** selects one series per anatomical plane, sorts DICOM slices by orientation-aware physical position, creates center-slice triplets, trains a study-level multi-label CNN, and writes `/kaggle/working/submission.csv`.
3. **Stage 3** independently reloads and validates the saved artifacts.

## Safety and reproducibility

- Uses only competition data and preinstalled Kaggle packages.
- Does not print study identifiers, reports, patient sex, DICOM pixels, or submission rows.
- Masks missing targets and uses every study with official labels.
- Preserves runtime sample-submission row order and the exact 12-target contract.
- Uses bounded uint8 caching in `/kaggle/temp`; only `submission.csv` is preserved as output.
- Falls back to clipped training prevalence only for unreadable or time-limited test studies.

## Verified browser run

- Accelerator: Tesla T4
- Officially labelled studies: 58 (49 train / 9 validation)
- Image size: 160 × 160
- Epochs: 3
- Best observed validation macro AUC: 0.5723 across all 12 targets
- Test fallback studies: 0
- Final prediction range: 0.422943 to 0.659098
- Final output: 3 rows × 13 columns with finite probabilities in [0, 1]

The visible test set has only three studies. Kaggle replaces it with the hidden test set during a saved competition run; the code derives IDs and output order from the runtime sample submission.


In [ ]:
# ============================================================
# RSNA Knee Abnormality Detection — Stage 1: data contract smoke
# ============================================================
# This cell intentionally avoids recursively listing hundreds of
# thousands of DICOM files. It reads only the small CSV metadata.

from pathlib import Path
import platform

import numpy as np
import pandas as pd
import pydicom
import sklearn
import torch

DATA_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ID_COLUMN = "StudyInstanceUID"
TARGET_COLUMNS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

required_files = {
    "train": DATA_ROOT / "train.csv",
    "train_series": DATA_ROOT / "train_series.csv",
    "test": DATA_ROOT / "test.csv",
    "test_series": DATA_ROOT / "test_series.csv",
    "sample_submission": DATA_ROOT / "sample_submission.csv",
}
missing_files = [name for name, path in required_files.items() if not path.exists()]
assert not missing_files, f"Missing competition files: {missing_files}"

train_df = pd.read_csv(required_files["train"])
train_series_df = pd.read_csv(required_files["train_series"])
test_df = pd.read_csv(required_files["test"])
test_series_df = pd.read_csv(required_files["test_series"])
sample_submission_df = pd.read_csv(required_files["sample_submission"])

assert ID_COLUMN in train_df.columns
assert ID_COLUMN in train_series_df.columns
assert ID_COLUMN in sample_submission_df.columns
assert list(sample_submission_df.columns) == [ID_COLUMN, *TARGET_COLUMNS]
assert len(TARGET_COLUMNS) == 12

print("Environment")
print(f"  Python: {platform.python_version()}")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print()
print("Competition metadata")
print(f"  train.csv: {train_df.shape}")
print(f"  train_series.csv: {train_series_df.shape}")
print(f"  test.csv: {test_df.shape}")
print(f"  test_series.csv: {test_series_df.shape}")
print(f"  sample_submission.csv: {sample_submission_df.shape}")
print(f"  target columns: {len(TARGET_COLUMNS)}")
print()
print("STAGE 1 PASSED: environment, files, and exact 12-target schema are valid.")

In [ ]:
# ============================================================
# RSNA Knee Abnormality Detection — Stage 2: 2.5D CNN pipeline
# ============================================================
# RUN_MODE controls resource use:
#   smoke    -> 24 studies, 128px images, 1 epoch
#   baseline -> every officially labelled study, 160px images, 3 epochs
#
# The smoke run is executed first. Change RUN_MODE to "baseline"
# only after every assertion and output check below passes.

from dataclasses import dataclass
from pathlib import Path
import hashlib
import random
import time
import warnings

import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

RUN_MODE = "baseline"
SEED = 42
RUN_STARTED = time.time()


def select_compute_backend(cuda_available, capability=None, compiled_arches=()):
    """Return cuda only when this PyTorch binary contains the visible GPU architecture."""
    if not cuda_available or capability is None:
        return "cpu"
    device_arch = f"sm_{capability[0]}{capability[1]}"
    return "cuda" if device_arch in set(compiled_arches) else "cpu"


def inspect_compute_backend():
    if not torch.cuda.is_available():
        return False, "CUDA is not available; using the CPU-safe profile."

    capability = torch.cuda.get_device_capability(0)
    compiled_arches = list(torch.cuda.get_arch_list())
    device_arch = f"sm_{capability[0]}{capability[1]}"
    gpu_name = torch.cuda.get_device_name(0)
    backend = select_compute_backend(True, capability, compiled_arches)

    if backend != "cuda":
        return (
            False,
            f"{gpu_name} uses {device_arch}, but this PyTorch build supports "
            f"{compiled_arches}; using the CPU-safe profile.",
        )
    return True, f"{gpu_name} ({device_arch})"


USE_CUDA, DEVICE_STATUS = inspect_compute_backend()


@dataclass(frozen=True)
class RunConfig:
    mode: str
    image_size: int
    batch_size: int
    epochs: int
    learning_rate: float
    validation_fraction: float
    max_train_studies: int | None
    max_series_per_study: int
    num_workers: int

    @classmethod
    def smoke(cls):
        return cls("smoke", 128, 4, 1, 1e-3, 0.25, 24, 3, 0)

    @classmethod
    def baseline(cls):
        # The GPU profile uses all labelled studies while keeping a
        # conservative margin below the nine-hour notebook limit.
        if USE_CUDA:
            return cls("baseline", 160, 24, 3, 5e-4, 0.15, None, 3, 2)
        return cls("baseline", 128, 16, 2, 1e-3, 0.15, None, 3, 2)


config = RunConfig.smoke() if RUN_MODE == "smoke" else RunConfig.baseline()
assert config.mode in {"smoke", "baseline"}
assert config.image_size > 0 and config.batch_size > 0 and config.epochs > 0
assert 0.0 < config.validation_fraction < 1.0
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if USE_CUDA:
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device("cuda" if USE_CUDA else "cpu")
print(f"Run mode: {config.mode}")
print(f"Device: {device}")
print(f"Device status: {DEVICE_STATUS}")
print(f"Configuration: {config}")


# ------------------------------------------------------------
# 1. Metadata preparation and series selection
# ------------------------------------------------------------

for frame in (train_df, train_series_df, test_df, test_series_df, sample_submission_df):
    frame[ID_COLUMN] = frame[ID_COLUMN].astype(str)

train_labels = train_df.copy()
for column in TARGET_COLUMNS:
    train_labels[column] = pd.to_numeric(train_labels[column], errors="coerce")

eligible_mask = train_labels[TARGET_COLUMNS].notna().any(axis=1)
eligible_train = train_labels.loc[eligible_mask].reset_index(drop=True)
print(f"Officially labelled studies: {len(eligible_train)}")
print(f"Unlabelled metadata/report studies excluded from supervised loss: {len(train_labels) - len(eligible_train)}")
sample_count = (
    len(eligible_train)
    if config.max_train_studies is None
    else min(config.max_train_studies, len(eligible_train))
)
selected_train = eligible_train.sample(n=sample_count, random_state=SEED).reset_index(drop=True)


def balanced_holdout_indices(label_frame, validation_fraction, seed, attempts=256):
    """Choose a deterministic split that preserves observed 0/1 coverage."""
    values = label_frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)
    observed = np.isfinite(values)
    positives = observed & (values >= 0.5)
    negatives = observed & (values < 0.5)
    count = len(label_frame)
    validation_count = max(1, int(round(count * validation_fraction)))
    rng = np.random.default_rng(seed)
    best = None
    best_score = np.inf

    for _ in range(attempts):
        permutation = rng.permutation(count)
        valid = permutation[:validation_count]
        train = permutation[validation_count:]

        train_pos = positives[train].sum(axis=0)
        valid_pos = positives[valid].sum(axis=0)
        train_neg = negatives[train].sum(axis=0)
        valid_neg = negatives[valid].sum(axis=0)
        total_pos = positives.sum(axis=0)
        total_neg = negatives.sum(axis=0)

        feasible_pos = total_pos >= 2
        feasible_neg = total_neg >= 2
        missing_class_penalty = (
            ((train_pos == 0) | (valid_pos == 0)) & feasible_pos
        ).sum() + (
            ((train_neg == 0) | (valid_neg == 0)) & feasible_neg
        ).sum()

        expected_pos = total_pos * validation_fraction
        expected_neg = total_neg * validation_fraction
        balance_error = np.abs(valid_pos - expected_pos).sum()
        balance_error += np.abs(valid_neg - expected_neg).sum()
        score = float(missing_class_penalty * 1_000_000 + balance_error)

        if score < best_score:
            best_score = score
            best = (train, valid)
            if missing_class_penalty == 0 and balance_error == 0:
                break

    assert best is not None
    return best


train_indices, valid_indices = balanced_holdout_indices(
    selected_train,
    config.validation_fraction,
    SEED,
)
train_subset = selected_train.iloc[train_indices].reset_index(drop=True)
valid_subset = selected_train.iloc[valid_indices].reset_index(drop=True)


def flag_priority(value):
    value = str(value).strip().lower()
    return int(value in {"1", "1.0", "true", "yes", "y"})


def build_series_lookup(series_frame, allowed_studies, max_series):
    filtered = series_frame[series_frame[ID_COLUMN].isin(set(allowed_studies))].copy()
    filtered["_fluid"] = filtered["Fluid_Sensitive"].map(flag_priority)
    filtered["_fat"] = filtered["Fat_Suppression"].map(flag_priority)
    lookup = {}

    for study_id, group in filtered.groupby(ID_COLUMN, sort=False):
        ordered = group.sort_values(
            ["_fluid", "_fat", "Anatomical_Plane"],
            ascending=[False, False, True],
            kind="stable",
        )
        chosen = []
        seen_planes = set()

        # Prefer one high-priority series from each anatomical plane.
        for row in ordered.itertuples(index=False):
            plane = str(getattr(row, "Anatomical_Plane", "unknown")).strip().lower()
            series_id = str(getattr(row, "SeriesInstanceUID"))
            if plane not in seen_planes:
                chosen.append(series_id)
                seen_planes.add(plane)
            if len(chosen) == max_series:
                break

        # Fill any remaining slots with the next best unused series.
        if len(chosen) < max_series:
            for series_id in ordered["SeriesInstanceUID"].astype(str):
                if series_id not in chosen:
                    chosen.append(series_id)
                if len(chosen) == max_series:
                    break

        lookup[str(study_id)] = chosen

    return lookup


selected_studies = selected_train[ID_COLUMN].tolist()
train_series_lookup = build_series_lookup(
    train_series_df,
    selected_studies,
    config.max_series_per_study,
)
test_series_lookup = build_series_lookup(
    test_series_df,
    test_df[ID_COLUMN].tolist(),
    config.max_series_per_study,
)

assert len(train_subset) > 0 and len(valid_subset) > 0
assert train_subset[ID_COLUMN].is_unique and valid_subset[ID_COLUMN].is_unique
assert set(train_subset[ID_COLUMN]).isdisjoint(set(valid_subset[ID_COLUMN]))
assert set(selected_studies).issubset(set(train_series_lookup))
assert set(test_df[ID_COLUMN]).issubset(set(test_series_lookup))

for target in TARGET_COLUMNS:
    observed_all = selected_train[target].dropna()
    if observed_all.nunique() >= 2 and len(observed_all) >= 4:
        assert train_subset[target].dropna().nunique() >= 2
        assert valid_subset[target].dropna().nunique() >= 2

print(f"Training studies: {len(train_subset)}")
print(f"Validation studies: {len(valid_subset)}")
print(f"Test studies: {len(test_df)}")


# ------------------------------------------------------------
# 2. DICOM decoding and center-triplet sampling
# ------------------------------------------------------------

TRAIN_IMAGE_ROOT = DATA_ROOT / "train_series"
TEST_IMAGE_ROOT = DATA_ROOT / "test_series"
CACHE_ROOT = Path("/kaggle/temp") / f"rsna_triplets_{config.image_size}px"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRAINING_DEADLINE = RUN_STARTED + 5.5 * 60 * 60
INFERENCE_DEADLINE = RUN_STARTED + 7.75 * 60 * 60


def dicom_sort_key(path):
    """Sort slices by position projected onto their orientation normal."""
    try:
        header = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        position = getattr(header, "ImagePositionPatient", None)
        orientation = getattr(header, "ImageOrientationPatient", None)
        if position is not None and orientation is not None:
            position_vector = np.asarray(position, dtype=np.float64)
            orientation_vector = np.asarray(orientation, dtype=np.float64)
            if position_vector.size >= 3 and orientation_vector.size >= 6:
                row_direction = orientation_vector[:3]
                column_direction = orientation_vector[3:6]
                slice_normal = np.cross(row_direction, column_direction)
                if np.linalg.norm(slice_normal) > 0:
                    location = float(np.dot(position_vector[:3], slice_normal))
                    return (0, location, path.name)
        instance = getattr(header, "InstanceNumber", None)
        if instance is not None:
            return (1, float(instance), path.name)
    except Exception:
        pass
    return (2, 0.0, path.name)


def ordered_dicom_paths(series_dir):
    paths = list(series_dir.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices found in {series_dir}")
    return sorted(paths, key=dicom_sort_key)


def decode_slice(path, image_size):
    dataset = pydicom.dcmread(path, force=True)
    image = dataset.pixel_array.astype(np.float32)
    if image.ndim == 3:
        image = image[len(image) // 2]
    if image.ndim != 2:
        raise ValueError(f"Unsupported pixel array shape: {image.shape}")
    image = image * float(getattr(dataset, "RescaleSlope", 1.0))
    image = image + float(getattr(dataset, "RescaleIntercept", 0.0))

    if getattr(dataset, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() - image

    finite = image[np.isfinite(image)]
    if finite.size == 0:
        image = np.zeros_like(image, dtype=np.float32)
    else:
        low, high = np.percentile(finite, [1.0, 99.0])
        if high <= low:
            image = np.zeros_like(image, dtype=np.float32)
        else:
            image = np.clip((image - low) / (high - low), 0.0, 1.0)

    image_u8 = (image * 255.0).astype(np.uint8)
    resized = Image.fromarray(image_u8).resize(
        (image_size, image_size),
        Image.Resampling.BILINEAR,
    )
    return np.asarray(resized, dtype=np.float32) / 255.0


def load_center_triplet(series_dir, image_size):
    paths = ordered_dicom_paths(series_dir)
    center = len(paths) // 2
    indices = [
        max(0, center - 1),
        center,
        min(len(paths) - 1, center + 1),
    ]
    channels = [decode_slice(paths[index], image_size) for index in indices]
    triplet = np.stack(channels, axis=0)
    if not np.isfinite(triplet).all():
        raise ValueError(f"Non-finite pixels in {series_dir}")
    return torch.from_numpy(triplet)


def load_cached_triplet(study_id, series_id, series_dir, image_size):
    cache_name = hashlib.sha1(
        f"{study_id}|{series_id}|{image_size}".encode("utf-8")
    ).hexdigest() + ".npy"
    cache_path = CACHE_ROOT / cache_name

    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        if cached.shape == (3, image_size, image_size) and cached.dtype == np.uint8:
            return torch.from_numpy(cached.astype(np.float32) / 255.0)

    triplet = load_center_triplet(series_dir, image_size)
    cached = np.rint(triplet.numpy() * 255.0).clip(0, 255).astype(np.uint8)
    np.save(cache_path, cached, allow_pickle=False)
    return torch.from_numpy(cached.astype(np.float32) / 255.0)


class KneeStudyDataset(Dataset):
    def __init__(self, studies, series_lookup, image_root, image_size, with_labels):
        self.studies = studies.reset_index(drop=True)
        self.series_lookup = series_lookup
        self.image_root = image_root
        self.image_size = image_size
        self.with_labels = with_labels

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, index):
        row = self.studies.iloc[index]
        study_id = str(row[ID_COLUMN])
        views = []

        for series_id in self.series_lookup.get(study_id, []):
            series_dir = self.image_root / study_id / series_id
            try:
                views.append(
                    load_cached_triplet(
                        study_id,
                        series_id,
                        series_dir,
                        self.image_size,
                    )
                )
            except Exception:
                continue

        decoded_views = len(views)
        if not views:
            views = [torch.zeros(3, self.image_size, self.image_size)]

        result = {
            "study_id": study_id,
            "images": torch.stack(views),
            "decoded_views": decoded_views,
        }

        if self.with_labels:
            labels = row[TARGET_COLUMNS].to_numpy(dtype=np.float32)
            mask = np.isfinite(labels)
            result["labels"] = torch.from_numpy(np.nan_to_num(labels, nan=0.0))
            result["label_mask"] = torch.from_numpy(mask.astype(np.float32))

        return result


def collate_studies(batch):
    batch_size = len(batch)
    max_views = max(item["images"].shape[0] for item in batch)
    channels, height, width = batch[0]["images"].shape[1:]

    images = torch.zeros(batch_size, max_views, channels, height, width)
    view_mask = torch.zeros(batch_size, max_views)
    study_ids = []

    for index, item in enumerate(batch):
        view_count = item["images"].shape[0]
        images[index, :view_count] = item["images"]
        view_mask[index, :view_count] = 1.0
        study_ids.append(item["study_id"])

    collated = {
        "study_ids": study_ids,
        "images": images,
        "view_mask": view_mask,
        "decoded_views": torch.tensor(
            [item["decoded_views"] for item in batch],
            dtype=torch.int64,
        ),
    }
    if "labels" in batch[0]:
        collated["labels"] = torch.stack([item["labels"] for item in batch])
        collated["label_mask"] = torch.stack([item["label_mask"] for item in batch])
    return collated


train_dataset = KneeStudyDataset(
    train_subset, train_series_lookup, TRAIN_IMAGE_ROOT, config.image_size, True
)
valid_dataset = KneeStudyDataset(
    valid_subset, train_series_lookup, TRAIN_IMAGE_ROOT, config.image_size, True
)
test_dataset = KneeStudyDataset(
    test_df, test_series_lookup, TEST_IMAGE_ROOT, config.image_size, False
)

# Browser-side pipeline tests on real competition DICOM data.
sample_item = train_dataset[0]
assert sample_item["images"].ndim == 4
assert sample_item["images"].shape[1:] == (3, config.image_size, config.image_size)
assert torch.isfinite(sample_item["images"]).all()
assert sample_item["labels"].shape == (12,)
assert sample_item["label_mask"].shape == (12,)
print(f"Real DICOM tensor test passed: {tuple(sample_item['images'].shape)}")


# ------------------------------------------------------------
# 3. Compact 2.5D CNN and masked multi-label loss
# ------------------------------------------------------------

class CompactKneeCNN(nn.Module):
    def __init__(self, num_targets=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(4, 16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, 32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, num_targets)

    def forward(self, images, view_mask):
        batch_size, view_count, channels, height, width = images.shape
        flat_images = images.reshape(batch_size * view_count, channels, height, width)
        features = self.encoder(flat_images).flatten(1)
        view_logits = self.classifier(features).reshape(batch_size, view_count, -1)
        weights = view_mask.unsqueeze(-1)
        return (view_logits * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)


def masked_bce_loss(logits, labels, label_mask, positive_weights):
    losses = nn.functional.binary_cross_entropy_with_logits(
        logits,
        labels,
        reduction="none",
        pos_weight=positive_weights,
    )
    return (losses * label_mask).sum() / label_mask.sum().clamp_min(1.0)


training_values = train_subset[TARGET_COLUMNS].to_numpy(dtype=np.float32)
training_observed = np.isfinite(training_values)
training_positive = ((training_values >= 0.5) & training_observed).sum(axis=0)
training_negative = ((training_values < 0.5) & training_observed).sum(axis=0)
weight_values = np.ones(len(TARGET_COLUMNS), dtype=np.float32)
has_both_classes = (training_positive > 0) & (training_negative > 0)
weight_values[has_both_classes] = np.clip(
    training_negative[has_both_classes] / training_positive[has_both_classes],
    0.5,
    10.0,
)
positive_weights = torch.tensor(weight_values, device=device)

model = CompactKneeCNN(num_targets=len(TARGET_COLUMNS)).to(device)
test_batch = collate_studies([sample_item])
with torch.no_grad():
    test_logits = model(
        test_batch["images"].to(device),
        test_batch["view_mask"].to(device),
    )
    test_loss = masked_bce_loss(
        test_logits,
        test_batch["labels"].to(device),
        test_batch["label_mask"].to(device),
        positive_weights,
    )
assert test_logits.shape == (1, 12)
assert torch.isfinite(test_logits).all()
assert torch.isfinite(test_loss)
print("Model forward-pass and masked-loss tests passed.")


# ------------------------------------------------------------
# 4. Training and validation
# ------------------------------------------------------------

worker_options = {
    "num_workers": config.num_workers,
    "persistent_workers": config.num_workers > 0,
}
if config.num_workers > 0:
    worker_options["prefetch_factor"] = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    pin_memory=USE_CUDA,
    collate_fn=collate_studies,
    **worker_options,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=USE_CUDA,
    collate_fn=collate_studies,
    **worker_options,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=USE_CUDA,
    collate_fn=collate_studies,
    **worker_options,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=1e-4,
)


def run_training_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    batches = 0

    for batch in loader:
        images = batch["images"].to(device, non_blocking=USE_CUDA)
        view_mask = batch["view_mask"].to(device, non_blocking=USE_CUDA)
        labels = batch["labels"].to(device, non_blocking=USE_CUDA)
        label_mask = batch["label_mask"].to(device, non_blocking=USE_CUDA)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images, view_mask)
        loss = masked_bce_loss(logits, labels, label_mask, positive_weights)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        batches += 1

    return total_loss / max(batches, 1)


def validate_model(model, loader):
    model.eval()
    all_labels = []
    all_masks = []
    all_probabilities = []
    total_loss = 0.0
    batches = 0

    with torch.no_grad():
        for batch in loader:
            images = batch["images"].to(device, non_blocking=USE_CUDA)
            view_mask = batch["view_mask"].to(device, non_blocking=USE_CUDA)
            labels = batch["labels"].to(device, non_blocking=USE_CUDA)
            label_mask = batch["label_mask"].to(device, non_blocking=USE_CUDA)

            logits = model(images, view_mask)
            total_loss += float(masked_bce_loss(logits, labels, label_mask, positive_weights).item())
            batches += 1
            all_labels.append(labels.cpu().numpy())
            all_masks.append(label_mask.cpu().numpy().astype(bool))
            all_probabilities.append(torch.sigmoid(logits).cpu().numpy())

    labels = np.concatenate(all_labels)
    masks = np.concatenate(all_masks)
    probabilities = np.concatenate(all_probabilities)
    target_aucs = []

    for target_index in range(len(TARGET_COLUMNS)):
        valid = masks[:, target_index]
        y_true = labels[valid, target_index]
        y_score = probabilities[valid, target_index]
        if len(y_true) >= 2 and np.unique(y_true).size == 2:
            target_aucs.append(roc_auc_score(y_true, y_score))

    macro_auc = float(np.mean(target_aucs)) if target_aucs else float("nan")
    return total_loss / max(batches, 1), macro_auc, len(target_aucs)


best_state = None
best_score = -np.inf
training_started = time.time()

for epoch in range(1, config.epochs + 1):
    train_loss = run_training_epoch(model, train_loader, optimizer)
    valid_loss, macro_auc, scored_targets = validate_model(model, valid_loader)
    score_for_selection = macro_auc if np.isfinite(macro_auc) else -valid_loss

    if score_for_selection > best_score:
        best_score = score_for_selection
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

    print(
        f"Epoch {epoch:02d}/{config.epochs}: "
        f"train_loss={train_loss:.4f}, "
        f"valid_loss={valid_loss:.4f}, "
        f"macro_auc={macro_auc:.4f}, "
        f"scored_targets={scored_targets}"
    )

assert best_state is not None
model.load_state_dict(best_state)
model.to(device)
elapsed_minutes = (time.time() - training_started) / 60.0
print(f"Training completed in {elapsed_minutes:.2f} minutes.")


# ------------------------------------------------------------
# 5. Test inference and strict submission validation
# ------------------------------------------------------------

def predict_studies(model, loader):
    model.eval()
    study_ids = []
    predictions = []
    fallback_ids = []

    with torch.inference_mode():
        for batch in loader:
            if time.time() >= INFERENCE_DEADLINE:
                break
            images = batch["images"].to(device, non_blocking=USE_CUDA)
            view_mask = batch["view_mask"].to(device, non_blocking=USE_CUDA)
            probabilities = torch.sigmoid(model(images, view_mask)).cpu().numpy()
            study_ids.extend(batch["study_ids"])
            predictions.append(probabilities)
            fallback_ids.extend(
                study_id
                for study_id, decoded_views in zip(
                    batch["study_ids"],
                    batch["decoded_views"].tolist(),
                )
                if decoded_views == 0
            )

    if predictions:
        prediction_values = np.concatenate(predictions, axis=0)
    else:
        prediction_values = np.empty((0, len(TARGET_COLUMNS)), dtype=np.float32)
    return study_ids, prediction_values, fallback_ids


prediction_ids, prediction_values, decode_fallback_ids = predict_studies(model, test_loader)
prediction_frame = pd.DataFrame(prediction_values, columns=TARGET_COLUMNS)
prediction_frame.insert(0, ID_COLUMN, prediction_ids)

submission = sample_submission_df[[ID_COLUMN]].merge(
    prediction_frame,
    on=ID_COLUMN,
    how="left",
    validate="one_to_one",
)

# A deterministic fallback is safer than an invalid missing row.
fallback_values = (
    selected_train[TARGET_COLUMNS]
    .mean(axis=0, skipna=True)
    .fillna(0.5)
    .clip(0.0, 1.0)
)
fallback_id_set = set(decode_fallback_ids)
for target in TARGET_COLUMNS:
    submission.loc[
        submission[ID_COLUMN].isin(fallback_id_set),
        target,
    ] = float(fallback_values[target])
    submission[target] = submission[target].fillna(float(fallback_values[target]))

fallback_count = int(
    submission[ID_COLUMN].isin(fallback_id_set).sum()
    + submission[ID_COLUMN].isin(set(sample_submission_df[ID_COLUMN]) - set(prediction_ids)).sum()
)

assert list(submission.columns) == [ID_COLUMN, *TARGET_COLUMNS]
assert len(submission) == len(sample_submission_df)
assert submission[ID_COLUMN].tolist() == sample_submission_df[ID_COLUMN].tolist()
assert submission[ID_COLUMN].is_unique
assert np.isfinite(submission[TARGET_COLUMNS].to_numpy()).all()
assert submission[TARGET_COLUMNS].to_numpy().min() >= 0.0
assert submission[TARGET_COLUMNS].to_numpy().max() <= 1.0

submission_path = Path("/kaggle/working/submission.csv")
temporary_submission_path = Path("/kaggle/working/submission.tmp")
checkpoint_path = Path(f"/kaggle/temp/rsna_knee_{config.mode}.pt")
submission.to_csv(temporary_submission_path, index=False)
temporary_submission_path.replace(submission_path)
torch.save(
    {
        "mode": config.mode,
        "model_state": best_state,
        "target_columns": TARGET_COLUMNS,
        "image_size": config.image_size,
        "seed": SEED,
    },
    checkpoint_path,
)

print()
print(f"Submission shape: {submission.shape}")
print(f"Prediction range: {submission[TARGET_COLUMNS].to_numpy().min():.6f} to "
      f"{submission[TARGET_COLUMNS].to_numpy().max():.6f}")
print(f"Fallback studies: {fallback_count}")
print(f"Saved: {submission_path}")
print(f"Temporary checkpoint: {checkpoint_path}")
if config.mode == "baseline" and len(submission) > 1:
    assert fallback_count < len(submission), "All test studies fell back; baseline is invalid."
    assert submission[TARGET_COLUMNS].std(axis=0).max() > 1e-6, "Predictions are constant."
print("STAGE 2 PASSED: DICOM, model, training, inference, and submission checks passed.")

In [ ]:
# ============================================================
# RSNA Knee Abnormality Detection — Stage 3: output verification
# ============================================================
# This cell independently reloads the files produced by Stage 2.

from pathlib import Path
import hashlib

verified_submission_path = Path("/kaggle/working/submission.csv")
verified_checkpoint_path = Path(f"/kaggle/temp/rsna_knee_{config.mode}.pt")

assert verified_submission_path.exists()
assert verified_checkpoint_path.exists()
assert verified_submission_path.stat().st_size > 0
assert verified_checkpoint_path.stat().st_size > 0

verified_submission = pd.read_csv(verified_submission_path)
verified_checkpoint = torch.load(
    verified_checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

assert list(verified_submission.columns) == [ID_COLUMN, *TARGET_COLUMNS]
assert len(verified_submission) == len(sample_submission_df)
assert verified_submission[ID_COLUMN].astype(str).tolist() == sample_submission_df[ID_COLUMN].astype(str).tolist()
assert verified_submission[ID_COLUMN].is_unique
assert verified_submission[TARGET_COLUMNS].apply(
    lambda column: pd.api.types.is_numeric_dtype(column)
).all()
assert np.isfinite(verified_submission[TARGET_COLUMNS].to_numpy()).all()
assert verified_submission[TARGET_COLUMNS].ge(0.0).all().all()
assert verified_submission[TARGET_COLUMNS].le(1.0).all().all()

assert verified_checkpoint["mode"] == config.mode
assert verified_checkpoint["target_columns"] == TARGET_COLUMNS
assert verified_checkpoint["image_size"] == config.image_size
assert len(verified_checkpoint["model_state"]) > 0

submission_sha256 = hashlib.sha256(verified_submission_path.read_bytes()).hexdigest()
print(f"Verified submission rows: {len(verified_submission)}")
print(f"Verified submission columns: {len(verified_submission.columns)}")
print(f"Submission file size: {verified_submission_path.stat().st_size} bytes")
print(f"Checkpoint file size: {verified_checkpoint_path.stat().st_size} bytes")
print(f"Submission SHA-256: {submission_sha256}")
print("STAGE 3 PASSED: saved artifacts reload correctly and satisfy the strict contract.")


In [ ]:
# Regression check: visible CUDA is not enough when this PyTorch build lacks the GPU architecture.
assert select_compute_backend(
    cuda_available=True,
    capability=(6, 0),
    compiled_arches=["sm_70", "sm_75", "sm_80"],
) == "cpu"
assert select_compute_backend(
    cuda_available=True,
    capability=(7, 5),
    compiled_arches=["sm_70", "sm_75", "sm_80"],
) == "cuda"
print("DEVICE SELECTION REGRESSION PASSED")